# What Happens to Your Data Inside an RNN and an LSTM Layer
### A toy example, explained end to end

In this notebook I take **2 tiny sentences** and follow them all the way through an RNN layer and an LSTM layer.

Each section has two parts. The **explanation** (in text) tells you exactly what goes in, what the layer does, and what comes out, with the actual numbers. Then a short **Keras cell** runs the real layer. Run it and compare what it prints with the numbers written above it. They should match.

To make the numbers easy to follow, I've set all the weights to small fixed values instead of random ones. In a real model, training would choose these values; the way the data flows stays exactly the same.

## The vocabulary you need

| Word | Meaning | In our toy example |
|---|---|---|
| **Batch** | how many sentences go into the layer together | 2 |
| **Time steps** | how many words each sentence has, after padding | 4 |
| **Features** | how many numbers describe one word | 2 |
| **Units** | how many numbers the layer uses as its memory (its size) | 2 |

Every RNN and LSTM layer in Keras takes input in this shape:

```
(batch, timesteps, features)   ->   here: (2, 4, 2)
```

Remember this one line and most of what follows is easy.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

np.set_printoptions(precision=4, suppress=True)

---
# Part 1: Preparing the input

## Step 1: Words become numbers

Our two sentences:

1. "movie was great"
2. "movie was not good"

Every word gets an ID. **0 is kept for padding**, so real words start at 1.

| Word | `<pad>` | movie | was | not | good | great | boring |
|---|---|---|---|---|---|---|---|
| **ID** | 0 | 1 | 2 | 3 | 4 | 5 | 6 |

So the sentences become:

```
"movie was great"      ->  [1, 2, 5]       (3 words)
"movie was not good"   ->  [1, 2, 3, 4]    (4 words)
```

## Step 2: Padding to the same length

The two lists have different lengths, but a layer needs a clean rectangle. So we make both of length **4** by adding 0 at the **front** (this is the Keras default, `padding='pre'`):

```
[[0, 1, 2, 5],
 [1, 2, 3, 4]]          shape (2, 4)  ->  (batch, timesteps)
```

Read it like a timetable. **Rows are sentences, columns are time steps.**

| | t = 0 | t = 1 | t = 2 | t = 3 |
|---|---|---|---|---|
| **Sentence 0** | `<pad>` | movie | was | great |
| **Sentence 1** | movie | was | not | good |

The RNN will walk across this table **from left to right**, one column at a time.

**Expected output of the next cell:** the matrix above, and shape `(2, 4)`.

In [2]:
sentences_as_ids = [[1, 2, 5],
                    [1, 2, 3, 4]]

padded = pad_sequences(sentences_as_ids, maxlen=4)
print(padded)
print("Shape:", padded.shape)

[[0 1 2 5]
 [1 2 3 4]]
Shape: (2, 4)


## Step 3: The Embedding layer, each ID becomes a small vector

A single ID like `5` doesn't tell the network anything about the word. The **Embedding layer** is a lookup table: row `k` holds a vector for word ID `k`. Here each word gets **2 numbers** (so `output_dim = 2`).

I've chosen the values so they're easy to read. The first number roughly means "how positive is this word".

| ID | Word | feature 1 | feature 2 |
|---|---|---|---|
| 0 | <pad> | 0 | 0 |
| 1 | movie | 0.1 | 0.2 |
| 2 | was | 0 | 0.1 |
| 3 | not | -0.5 | 0.3 |
| 4 | good | 0.8 | 0.1 |
| 5 | great | 0.9 | 0.2 |
| 6 | boring | -0.7 | -0.2 |

**What goes in:** `(2, 4)`, just IDs.
**What the layer does:** replaces every ID with its row from the table.
**What comes out:** `(2, 4, 2)`, which is `(batch, timesteps, features)`.

| | t = 0 | t = 1 | t = 2 | t = 3 |
|---|---|---|---|---|
| **Sentence 0** | <pad> [0.0, 0.0] | movie [0.1000, 0.2000] | was [0.0, 0.1000] | great [0.9000, 0.2000] |
| **Sentence 1** | movie [0.1000, 0.2000] | was [0.0, 0.1000] | not [-0.5000, 0.3000] | good [0.8000, 0.1000] |

This 3D block is exactly what the RNN and LSTM layers will receive. At each time step, the layer gets **one small vector of 2 numbers per sentence**.

**Expected output of the next cell:** shape `(2, 4, 2)` and the vectors in the table above.

In [3]:
embedding_table = np.array([
    [ 0.0,  0.0],   # 0 <pad>
    [ 0.1,  0.2],   # 1 movie
    [ 0.0,  0.1],   # 2 was
    [-0.5,  0.3],   # 3 not
    [ 0.8,  0.1],   # 4 good
    [ 0.9,  0.2],   # 5 great
    [-0.7, -0.2],   # 6 boring
])

embedding = Embedding(input_dim=7, output_dim=2)
embedding.build((None, 4))
embedding.set_weights([embedding_table])     # use our fixed table instead of random numbers

X = embedding(padded)
print("Shape:", X.shape)
print(X.numpy())

Shape: (2, 4, 2)
[[[ 0.   0. ]
  [ 0.1  0.2]
  [ 0.   0.1]
  [ 0.9  0.2]]

 [[ 0.1  0.2]
  [ 0.   0.1]
  [-0.5  0.3]
  [ 0.8  0.1]]]


---
# Part 2: The RNN layer (`SimpleRNN`)

## The idea

The RNN layer reads the sentence **one word at a time**, left to right. It carries a **memory vector** `h`, whose size is the number of **units** (here 2). The memory starts empty, `[0, 0]`.

At every time step it does one thing: **mix the current word with the previous memory** to make a new memory.

$$h_t = \tanh(\;x_t \cdot W_x \;+\; h_{t-1} \cdot W_h \;+\; b\;)$$

```
                x_t  (current word, 2 numbers)
                 |
                 v
  h_(t-1) --> [ RNN cell ] --> h_t  --> used at the next step
 (old memory)                (new memory, 2 numbers)
```

## The layer's weights, and why they have these shapes

`SimpleRNN(units=2)` receiving words with 2 features creates three sets of numbers:

| Keras name | Symbol | Shape | Why this shape | Job |
|---|---|---|---|---|
| `kernel` | $W_x$ | (2, 2) | (features, units): turns a word into memory-sized numbers | how the current word affects memory |
| `recurrent_kernel` | $W_h$ | (2, 2) | (units, units): turns old memory into new memory | how the old memory affects new memory |
| `bias` | $b$ | (2,) | one per unit | a constant nudge |

**Total parameters** = 2×2 + 2×2 + 2 = **10**. The general formula is `units × (features + units + 1)`.

The **same three weights are used at every time step**. The layer doesn't grow for longer sentences; it just loops more times.

Our fixed values:

$W_x$ =

| unit 1 | unit 2 |
|---|---|
| 0.5 | -0.3 |
| 0.2 | 0.4 |

$W_h$ =

| unit 1 | unit 2 |
|---|---|
| 0.1 | 0.2 |
| -0.1 | 0.3 |

$b$ = [0.0, 0.1]

## Following sentence 1, "movie was not good", step by step

Memory starts at $h_{-1}$ = [0, 0].

### Time step t = 0, word = "movie"

The cell receives $x_0$ = [0.1000, 0.2000] and the previous memory $h_{-1}$ = [0.0000, 0.0000].

1. Current word × input weights: `x · Wx` = [0.1×0.5 + 0.2×0.2,  0.1×(-0.3) + 0.2×0.4] = **[0.0900, 0.0500]**
2. Previous memory × memory weights: `h_prev · Wh` = [0×0.1 + 0×(-0.1),  0×0.2 + 0×0.3] = **[0.0000, 0.0000]**
3. Add everything, including the bias: [0.0900, 0.0500] + [0.0000, 0.0000] + [0.0000, 0.1000] = **[0.0900, 0.1500]**
4. Squash with tanh: tanh([0.0900, 0.1500]) = **[0.0898, 0.1489]**

New memory $h_0$ = **[0.0898, 0.1489]**. This goes to the next step, and is also the output for this step.

### Time step t = 1, word = "was"

The cell receives $x_1$ = [0.0000, 0.1000] and the previous memory $h_{0}$ = [0.0898, 0.1489].

1. Current word × input weights: `x · Wx` = [0×0.5 + 0.1×0.2,  0×(-0.3) + 0.1×0.4] = **[0.0200, 0.0400]**
2. Previous memory × memory weights: `h_prev · Wh` = [0.0897578×0.1 + 0.148885×(-0.1),  0.0897578×0.2 + 0.148885×0.3] = **[-0.0059, 0.0626]**
3. Add everything, including the bias: [0.0200, 0.0400] + [-0.0059, 0.0626] + [0.0000, 0.1000] = **[0.0141, 0.2026]**
4. Squash with tanh: tanh([0.0141, 0.2026]) = **[0.0141, 0.1999]**

New memory $h_1$ = **[0.0141, 0.1999]**. This goes to the next step, and is also the output for this step.

### Time step t = 2, word = "not"

The cell receives $x_2$ = [-0.5000, 0.3000] and the previous memory $h_{1}$ = [0.0141, 0.1999].

1. Current word × input weights: `x · Wx` = [(-0.5)×0.5 + 0.3×0.2,  (-0.5)×(-0.3) + 0.3×0.4] = **[-0.1900, 0.2700]**
2. Previous memory × memory weights: `h_prev · Wh` = [0.0140863×0.1 + 0.199889×(-0.1),  0.0140863×0.2 + 0.199889×0.3] = **[-0.0186, 0.0628]**
3. Add everything, including the bias: [-0.1900, 0.2700] + [-0.0186, 0.0628] + [0.0000, 0.1000] = **[-0.2086, 0.4328]**
4. Squash with tanh: tanh([-0.2086, 0.4328]) = **[-0.2056, 0.4076]**

New memory $h_2$ = **[-0.2056, 0.4076]**. This goes to the next step, and is also the output for this step.

### Time step t = 3, word = "good"

The cell receives $x_3$ = [0.8000, 0.1000] and the previous memory $h_{2}$ = [-0.2056, 0.4076].

1. Current word × input weights: `x · Wx` = [0.8×0.5 + 0.1×0.2,  0.8×(-0.3) + 0.1×0.4] = **[0.4200, -0.2000]**
2. Previous memory × memory weights: `h_prev · Wh` = [(-0.205607)×0.1 + 0.407645×(-0.1),  (-0.205607)×0.2 + 0.407645×0.3] = **[-0.0613, 0.0812]**
3. Add everything, including the bias: [0.4200, -0.2000] + [-0.0613, 0.0812] + [0.0000, 0.1000] = **[0.3587, -0.0188]**
4. Squash with tanh: tanh([0.3587, -0.0188]) = **[0.3440, -0.0188]**

New memory $h_3$ = **[0.3440, -0.0188]**. This goes to the next step, and is also the output for this step.

## The whole walk in one table (sentence 1)

| t | word | input $x_t$ | memory in $h_{t-1}$ | memory out $h_t$ |
|---|---|---|---|---|
| 0 | movie | [0.1000, 0.2000] | [0.0000, 0.0000] | [0.0898, 0.1489] |
| 1 | was | [0.0000, 0.1000] | [0.0898, 0.1489] | [0.0141, 0.1999] |
| 2 | not | [-0.5000, 0.3000] | [0.0141, 0.1999] | [-0.2056, 0.4076] |
| 3 | good | [0.8000, 0.1000] | [-0.2056, 0.4076] | [0.3440, -0.0188] |

Three things to notice:

**At t = 0 the memory is empty**, so only the word "movie" counts. From t = 1 onwards, the old memory is mixed in. That's how "movie" still has a small effect when the layer reaches "good".

**tanh keeps every value between -1 and 1**, so the memory never blows up.

**The final memory, [0.3440, -0.0188], is a 2-number summary of the whole sentence.**

## And sentence 0? Same thing, at the same time

The layer doesn't finish sentence 0 and then start sentence 1. At each time step it takes **that column for both sentences together**:

```
at time step t:
  input  x_t      : (2, 2)   2 sentences x 2 features
  memory h_(t-1)  : (2, 2)   2 sentences x 2 units
  output h_t      : (2, 2)   2 sentences x 2 units
```

**Time steps go one after another. Sentences in a batch go side by side.**

Sentence 0 starts with `<pad>`, whose vector is [0, 0]. So at t = 0 only the bias is left: tanh([0, 0.1]) = [0.0000, 0.0997]. **The memory is not zero after a padding step.** The layer treats padding like a real word unless you use `mask_zero=True` in the Embedding layer.

| t | Sentence 0 memory $h_t$ | Sentence 1 memory $h_t$ |
|---|---|---|
| 0 | [0.0000, 0.0997] | [0.0898, 0.1489] |
| 1 | [0.0799, 0.1780] | [0.0141, 0.1999] |
| 2 | [0.0102, 0.2064] | [-0.2056, 0.4076] |
| 3 | [0.4385, -0.0260] | [0.3440, -0.0188] |

## What comes back out of the layer

The layer calculated a memory at every time step. What it **hands to the next layer** depends on one setting:

**`return_sequences=True`** returns the memory at **every** step.
Shape `(batch, timesteps, units)` = **(2, 4, 2)**: the whole table above.
Use it when the next layer is another RNN or LSTM, because that layer needs a sequence to read.

**`return_sequences=False`** (the default) returns only the **last** memory.
Shape `(batch, units)` = **(2, 2)**: the last row of the table, [0.4385, -0.0260] and [0.3440, -0.0188].
Use it when the next layer is Dense, because you want one summary per sentence.

**Expected output of the next cell:** the shapes `(2, 4, 2)` and `(2, 2)` and the numbers in the table above.

In [ ]:
rnn_all_steps = SimpleRNN(units=2, return_sequences=True)
rnn_last_step = SimpleRNN(units=2, return_sequences=False)

rnn_weights = [np.array([[0.5, -0.3], [0.2, 0.4]]),     # kernel            Wx  (features, units)
               np.array([[0.1,  0.2], [-0.1, 0.3]]),    # recurrent_kernel  Wh  (units, units)
               np.array([0.0, 0.1])]                    # bias              b   (units,)

for layer in [rnn_all_steps, rnn_last_step]:
    layer.build((None, 4, 2))
    layer.set_weights(rnn_weights)

out_all = rnn_all_steps(X)
out_last = rnn_last_step(X)

print("return_sequences=True  -> shape", out_all.shape)
print(out_all.numpy())
print("\nreturn_sequences=False -> shape", out_last.shape)
print(out_last.numpy())
print("\nParameters in the layer:", rnn_all_steps.count_params())

---
# Part 3: The LSTM layer

## Why a different layer?

In the RNN, the whole memory is rewritten through tanh at **every** step. In a long sentence, words from the beginning slowly fade away. The LSTM fixes this by keeping **two** memories instead of one:

| Memory | Name | Role |
|---|---|---|
| `c` | **cell state** | long-term memory, a "conveyor belt" that changes only a little at each step |
| `h` | **hidden state** | short-term memory, and **the output** of the layer |

Both start as [0, 0], and both have size = **units** (here 2).

## What happens inside at every time step

The cell looks at the current word $x_t$ and the previous output $h_{t-1}$, and makes **four** calculations, each with its own weights:

| Part | Symbol | Activation | Gives values | Question it answers |
|---|---|---|---|---|
| Forget gate | $f$ | sigmoid | 0 to 1 | How much of the old long-term memory should I **keep**? |
| Input gate | $i$ | sigmoid | 0 to 1 | How much of the new information should I **write**? |
| Candidate | $g$ | tanh | -1 to 1 | **What** is the new information? |
| Output gate | $o$ | sigmoid | 0 to 1 | How much of the memory should I **show** as output? |

A gate value of 0 means "block completely" and 1 means "let it all through".

Then the two memories are updated:

$$c_t = f \times c_{t-1} \;+\; i \times g \qquad \text{(keep some old, add some new)}$$

$$h_t = o \times \tanh(c_t) \qquad \text{(decide how much to show)}$$

(× here means multiply number by number: first unit with first unit, second with second.)

## The layer's weights

`LSTM(units=2)` needs a separate set of weights for each of the 4 parts, so it has **4 times** the weights of a SimpleRNN:

| Keras name | Shape | What's inside |
|---|---|---|
| `kernel` | (features, 4 × units) = (2, 8) | $W_i, W_f, W_g, W_o$ side by side |
| `recurrent_kernel` | (units, 4 × units) = (2, 8) | $U_i, U_f, U_g, U_o$ side by side |
| `bias` | (4 × units,) = (8,) | $b_i, b_f, b_g, b_o$ side by side |

**Total parameters** = 16 + 16 + 8 = **40**. The general formula is `4 × units × (features + units + 1)`.

Keras keeps the four parts in the order **input, forget, candidate, output**. That's the order we'll use when we load our weights.

Our fixed values ($W$ multiplies the word, $U$ multiplies the previous output $h$):

| Part | $W$ (for $x_t$) | $U$ (for $h_{t-1}$) | bias |
|---|---|---|---|
| input $i$ | [[0.4, 0.1], [0.2, 0.3]] | [[0.1, 0], [0, 0.1]] | [0, 0] |
| forget $f$ | [[0.3, 0.2], [0.1, 0.4]] | [[0.2, 0.1], [0.1, 0.2]] | **[1, 1]** |
| candidate $g$ | [[0.6, -0.4], [0.2, 0.5]] | [[0.1, 0.1], [-0.1, 0.1]] | [0, 0] |
| output $o$ | [[0.3, 0.3], [0.2, -0.2]] | [[0.1, 0], [0, 0.1]] | [0, 0] |

The forget bias is **1**, which is what Keras uses by default. It makes a fresh LSTM lean towards *remembering*, since sigmoid(1) ≈ 0.73.

## Following sentence 1, "movie was not good"

### Time step t = 0, word = "movie", fully worked out

Inputs: $x_0$ = [0.1000, 0.2000], $h_{-1}$ = [0, 0], $c_{-1}$ = [0, 0].

Because $h_{-1}$ is zero, the $U$ part adds nothing at this step.

| Part | Symbol | Activation | $x \cdot W$ + $h \cdot U$ + bias | Result |
|---|---|---|---|---|
| Forget | f | sigmoid | [0.0500, 0.1000] + [0.0000, 0.0000] + [1.0000, 1.0000] = [1.0500, 1.1000] | **[0.7408, 0.7503]** |
| Input | i | sigmoid | [0.0800, 0.0700] + [0.0000, 0.0000] + [0.0000, 0.0000] = [0.0800, 0.0700] | **[0.5200, 0.5175]** |
| Candidate | g | tanh | [0.1000, 0.0600] + [0.0000, 0.0000] + [0.0000, 0.0000] = [0.1000, 0.0600] | **[0.0997, 0.0599]** |
| Output | o | sigmoid | [0.0700, -0.0100] + [0.0000, 0.0000] + [0.0000, 0.0000] = [0.0700, -0.0100] | **[0.5175, 0.4975]** |

Long-term memory:
$c_0$ = f × $c_{-1}$ + i × g = [0.7408, 0.7503] × [0, 0] + [0.5200, 0.5175] × [0.0997, 0.0599] = [0.0000, 0.0000] + [0.0518, 0.0310] = **[0.0518, 0.0310]**

Output:
$h_0$ = o × tanh($c_0$) = [0.5175, 0.4975] × [0.0518, 0.0310] = **[0.0268, 0.0154]**

### All four time steps

Every step repeats exactly the same recipe. Here are the results:

| t | word | forget $f$ | input $i$ | candidate $g$ | output $o$ | kept: $f \times c_{t-1}$ | added: $i \times g$ | memory $c_t$ | output $h_t$ |
|---|---|---|---|---|---|---|---|---|---|
| 0 | movie | [0.7408, 0.7503] | [0.5200, 0.5175] | [0.0997, 0.0599] | [0.5175, 0.4975] | [0.0000, 0.0000] | [0.0518, 0.0310] | [0.0518, 0.0310] | [0.0268, 0.0154] |
| 1 | was | [0.7344, 0.7400] | [0.5057, 0.5079] | [0.0211, 0.0542] | [0.5057, 0.4954] | [0.0381, 0.0229] | [0.0107, 0.0275] | [0.0487, 0.0505] | [0.0246, 0.0250] |
| 2 | not | [0.7084, 0.7364] | [0.4657, 0.5106] | [-0.2355, 0.3408] | [0.4781, 0.4483] | [0.0345, 0.0372] | [-0.1097, 0.1740] | [-0.0751, 0.2112] | [-0.0359, 0.0933] |
| 3 | good | [0.7777, 0.7712] | [0.5833, 0.5298] | [0.4519, -0.2583] | [0.5638, 0.5571] | [-0.0584, 0.1628] | [0.2636, -0.1368] | [0.2052, 0.0260] | [0.1141, 0.0145] |

How to read this table:

**Follow the $c_t$ column.** At each step it's just "keep a fraction of the old value, then add something". Nothing squashes it on the way, which is why information can survive many steps. This is the conveyor belt.

**At "not", the candidate $g$ turns negative in the first unit** (-0.2355), because "not" has a negative first feature. So the first unit of $c$ goes negative.

**At "good", the first unit gets a big positive write** (+0.2636), which flips it back to 0.2052. The final memory carries the effect of both words.

**$h_t$ is always smaller in size than $c_t$**, because it is $c_t$ squashed by tanh and then scaled down by the output gate. $c$ is the storage; $h$ is what the cell chooses to show.

## Both sentences together

Just like the RNN, the LSTM processes both sentences side by side at each time step.

| t | Sentence 0: $h_t$ | Sentence 0: $c_t$ | Sentence 1: $h_t$ | Sentence 1: $c_t$ |
|---|---|---|---|---|
| 0 | [0.0000, 0.0000] | [0.0000, 0.0000] | [0.0268, 0.0154] | [0.0518, 0.0310] |
| 1 | [0.0268, 0.0154] | [0.0518, 0.0310] | [0.0246, 0.0250] | [0.0487, 0.0505] |
| 2 | [0.0246, 0.0250] | [0.0487, 0.0505] | [-0.0359, 0.0933] | [-0.0751, 0.2112] |
| 3 | [0.1950, -0.0528] | [0.3515, -0.0950] | [0.1141, 0.0145] | [0.2052, 0.0260] |

Notice that for sentence 0 the `<pad>` step gives exactly [0, 0]. That's only because the word vector, the starting memory **and** the candidate bias are all zero here, so there's nothing to write. With other weights (like the RNN's non-zero bias earlier), a padding step does change the memory.

## What comes back out of the layer

The LSTM gives back **$h$**, never $c$ (unless you specifically ask for it). The same setting decides how much:

**`return_sequences=True`** returns $h_t$ at **every** step: shape **(2, 4, 2)**, the two $h_t$ columns above.

**`return_sequences=False`** (default) returns only the **last** $h_t$: shape **(2, 2)**, which is [0.1950, -0.0528] and [0.1141, 0.0145].

**`return_state=True`** additionally hands back the final $h$ and the final $c$. I use it in the next cell only so you can see $c$ as well.

**Expected output of the next cell:** shapes `(2, 4, 2)` and `(2, 2)`, and the $h_t$ and $c_t$ values in the table above.

In [ ]:
# The four parts are placed side by side in the Keras order: input, forget, candidate, output
kernel = np.array([[0.4, 0.1,   0.3, 0.2,   0.6, -0.4,   0.3,  0.3],
                   [0.2, 0.3,   0.1, 0.4,   0.2,  0.5,   0.2, -0.2]])      # (features=2, 4 x units=8)

recurrent_kernel = np.array([[0.1, 0.0,   0.2, 0.1,   0.1, 0.1,   0.1, 0.0],
                             [0.0, 0.1,   0.1, 0.2,  -0.1, 0.1,   0.0, 0.1]])  # (units=2, 8)

bias = np.array([0.0, 0.0,   1.0, 1.0,   0.0, 0.0,   0.0, 0.0])            # (8,)

lstm_all_steps = LSTM(units=2, return_sequences=True)
lstm_last_step = LSTM(units=2, return_state=True)       # returns: last h, last h again, last c

for layer in [lstm_all_steps, lstm_last_step]:
    layer.build((None, 4, 2))
    layer.set_weights([kernel, recurrent_kernel, bias])

out_all = lstm_all_steps(X)
last_h, _, last_c = lstm_last_step(X)

print("return_sequences=True -> shape", out_all.shape)
print(out_all.numpy())
print("\nLast h -> shape", last_h.shape)
print(last_h.numpy())
print("\nLast c (long-term memory) -> shape", last_c.shape)
print(last_c.numpy())
print("\nParameters in the layer:", lstm_all_steps.count_params())

---
# Part 4: From the last memory to a prediction

For sentiment classification we take the **last** output of the layer (`return_sequences=False`), shape `(2, 2)`, and pass it to `Dense(1, activation='sigmoid')`.

The Dense layer knows nothing about sentences or time steps. It just sees **2 numbers per sentence** and turns them into **1 probability**:

$$\text{prediction} = \text{sigmoid}(h_{\text{last}} \cdot W_y + b_y)$$

With $W_y$ = [[2], [1]] and $b_y$ = 0:

**After the RNN:**

| Sentence | last $h$ | $h \cdot W_y$ | sigmoid → P(positive) |
|---|---|---|---|
| 0 | [0.4385, -0.0260] | 2 × 0.4385 + 1 × (-0.026) = 0.8510 | **0.7008** |
| 1 | [0.3440, -0.0188] | 2 × 0.3440 + 1 × (-0.0188) = 0.6693 | **0.6613** |

**After the LSTM:**

| Sentence | last $h$ | $h \cdot W_y$ | sigmoid → P(positive) |
|---|---|---|---|
| 0 | [0.1950, -0.0528] | 2 × 0.1950 + 1 × (-0.0528) = 0.3372 | **0.5835** |
| 1 | [0.1141, 0.0145] | 2 × 0.1141 + 1 × 0.0145 = 0.2426 | **0.5604** |

These probabilities don't mean anything yet, because the weights were chosen by hand, not learned. **Training** is the process of adjusting the embedding table, the RNN/LSTM weights and the Dense weights until these probabilities match the real labels. The flow of data you've just followed stays exactly the same before and after training.

**Expected output of the next cell:** the two P(positive) columns above.

In [ ]:
dense = Dense(1, activation="sigmoid")
dense.build((None, 2))
dense.set_weights([np.array([[2.0], [1.0]]), np.array([0.0])])

print("After RNN :", dense(out_last).numpy().ravel())
print("After LSTM:", dense(last_h).numpy().ravel())

---
# Part 5: The full journey in one picture

```
"movie was great", "movie was not good"
          |
          |  words -> IDs, then padding
          v
IDs                          (2, 4)        batch, timesteps
          |
          |  Embedding: each ID -> 2 numbers
          v
Word vectors                 (2, 4, 2)     batch, timesteps, features
          |
          |  RNN / LSTM: read 4 steps, carry a memory of size 2
          v
return_sequences=True        (2, 4, 2)     batch, timesteps, units
return_sequences=False       (2, 2)        batch, units
          |
          |  Dense(1) + sigmoid
          v
Prediction                   (2, 1)        one probability per sentence
```

## RNN vs LSTM, side by side

| | SimpleRNN | LSTM |
|---|---|---|
| Receives at each step | word $x_t$ + memory $h_{t-1}$ | word $x_t$ + $h_{t-1}$ + long-term memory $c_{t-1}$ |
| Does at each step | one mix + tanh | 4 parts (forget, input, candidate, output), then updates $c$ and $h$ |
| Hands back | $h$ | $h$ ($c$ stays inside) |
| Output shape, `return_sequences=True` | (batch, timesteps, units) | (batch, timesteps, units) |
| Output shape, `return_sequences=False` | (batch, units) | (batch, units) |
| Parameters | units × (features + units + 1) = **10** here | 4 × units × (features + units + 1) = **40** here |
| Long sentences | early words fade | long-term memory carries them further |

## Practice questions

1. 32 reviews, each padded to 100 words, go into `Embedding(20000, 128)`. What shape comes out?
2. That goes into `SimpleRNN(50, return_sequences=True)`. What shape comes out?
3. Then `LSTM(100)`. What shape comes out, and how many parameters does it have?
4. Why is sentence 0's RNN memory not [0, 0] after the `<pad>` step?
5. If you stack two LSTM layers, which one needs `return_sequences=True`, and why?

*Answers: (1) (32, 100, 128). (2) (32, 100, 50). (3) (32, 100), and 4 × 100 × (50 + 100 + 1) = 60,400 parameters. (4) The bias is still added, and tanh(bias) is not zero. (5) The first one, because the second LSTM needs a full sequence to read, not a single summary.*